# 02 — GRU Classifier

Prepare sequence data and train a bidirectional GRU directly on raw
multi-band light curves. No hand-crafted features — the network learns
temporal patterns from the raw flux time series.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, os.path.abspath("../.."))
from config import DATA_CONFIG, MODEL_CONFIG, PREPROCESS_CONFIG
from src.preprocessing import batch_prepare_rnn
from src.models import LightCurveRNN, create_data_loaders, compute_class_weights, train_rnn
from src.metrics import (
    weighted_log_loss, macro_f1, plot_confusion_matrix, classification_summary,
)
from src.utils import encode_labels, get_class_name, plot_training_curves, save_figure

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
metadata = pd.read_parquet(os.path.join(DATA_CONFIG['processed_dir'], 'training_metadata.parquet'))
lc = pd.read_parquet(os.path.join(DATA_CONFIG['processed_dir'], 'training_lc.parquet'))
print(f"Loaded {len(metadata)} objects, {len(lc):,} observations")

## 1. Prepare RNN Input Sequences

Each object becomes a sequence of timesteps sorted by MJD across all bands.
Each timestep = (delta_t, flux, flux_err, passband_onehot x6) = 9 features.

In [ ]:
# This takes a few minutes
sequences, lengths, targets, object_ids = batch_prepare_rnn(lc, metadata)

print(f"Sequences shape: {sequences.shape}")  # (n_objects, max_len, 9)
print(f"Lengths: min={lengths.min()}, max={lengths.max()}, median={np.median(lengths):.0f}")
print(f"Objects: {len(object_ids)}, Classes: {np.unique(targets)}")

In [ ]:
# Encode labels
labels, encode_map, decode_map = encode_labels(targets)
class_names_ordered = [get_class_name(decode_map[i]) for i in range(len(decode_map))]
print(f"Encoded labels: {np.unique(labels)}")
print(f"Class distribution:")
for i in range(len(decode_map)):
    count = (labels == i).sum()
    print(f"  {i:2d} ({class_names_ordered[i]:15s}): {count}")

## 2. Create DataLoaders

In [ ]:
train_loader, val_loader, test_loader = create_data_loaders(sequences, lengths, labels)

print(f"Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

# Verify batch shape
x_sample, len_sample, y_sample = next(iter(train_loader))
print(f"Batch shape: x={x_sample.shape}, lengths={len_sample.shape}, y={y_sample.shape}")

## 3. Initialize Model

In [ ]:
model = LightCurveRNN()
print(model)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {n_params:,}")
print(f"Trainable parameters: {n_trainable:,}")

In [ ]:
# Compute class weights
class_weights = compute_class_weights(labels)
print("Class weights:")
for i, w in enumerate(class_weights):
    print(f"  {class_names_ordered[i]:15s}: {w:.3f}")

## 4. Train

In [ ]:
history = train_rnn(model, train_loader, val_loader,
                    class_weights=class_weights, device=device)
print(f"\nBest epoch: {history['best_epoch']}")

In [ ]:
# Training curves
fig = plot_training_curves(history)
save_figure(fig, 'rnn_training_curves')
plt.show()

## 5. Evaluate on Test Set

In [ ]:
model.eval()
all_preds, all_probs, all_true = [], [], []

with torch.no_grad():
    for x_batch, len_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        len_batch = len_batch.to(device)
        logits = model(x_batch, len_batch)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_probs.append(probs)
        all_true.append(y_batch.numpy())

y_test = np.concatenate(all_true)
y_pred = np.concatenate(all_preds)
y_pred_proba = np.concatenate(all_probs)

results = classification_summary(y_test, y_pred, y_pred_proba,
                                 class_names={i: class_names_ordered[i]
                                              for i in range(len(class_names_ordered))})

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(12, 10))
plot_confusion_matrix(y_test, y_pred, class_names_ordered, normalize=True, ax=ax)
ax.set_title('GRU — Confusion Matrix (Normalized)')
plt.tight_layout()
save_figure(fig, 'rnn_confusion_matrix')
plt.show()

## 6. Save Model and Predictions

In [ ]:
os.makedirs('../../data', exist_ok=True)
torch.save(model.state_dict(), '../../data/rnn_model.pt')
np.savez('../../data/rnn_test_preds.npz',
         y_true=y_test, y_pred=y_pred, y_pred_proba=y_pred_proba,
         class_names=class_names_ordered)
print("Saved model and predictions")